<!-- cabecera-entorno -->
## Antes de empezar

**Clase 10 · Visualización interactiva: Plotly + Streamlit** — Bloque 2 · Demo. Este cuaderno se
recorre **por su cuenta**: explica cada concepto antes de usarlo, y el profesor circula por el salón
resolviendo dudas. No hay que esperar a que alguien lo dicte desde el tablero.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import plotly.express as px
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/calidad_aire_risaralda.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 10 · Demo — Plotly y Streamlit

**Dataset:** `../datos/calidad_aire_risaralda.csv` — mediciones de material particulado en el aire de
Risaralda, vía datos.gov.co. 5.047 filas x 5 columnas.

## Cómo se recorre este cuaderno

**Usted avanza solo, leyendo.** Cada bloque de código viene precedido de la explicación del concepto
que usa, y cada término se define la primera vez que aparece. El profesor circula por el salón: si
algo no cuadra después de leer la explicación, levante la mano.

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. **Escribir código es el bloque 3**, con el reto, y
es lo que se entrega. Este cuaderno no tiene ejercicios ni verificador: no hay celdas verdes que
perseguir.

**Las doce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se responden
escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque plegable
*"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.** Abrirlo antes no
le ahorra nada: lo que se evalúa en el Momento 2 no es reproducir un `groupby`, es mirar un tablero y
decir qué decisión cambia.

**Los gráficos vienen ya titulados y con los ejes etiquetados.** Eso no es decoración: es el modelo a
copiar en el reto, que sí lo exige. Mire cómo está escrito cada `update_layout` antes de pasar a la
siguiente celda.

| Marca | Qué significa |
|-------|---------------|
| **Concepto** | Qué es y por qué se usa. Se lee antes de correr el código de abajo |
| **Para entender qué está pasando** | El fundamento. Se puede saltar hoy y volver después; nada más depende de él |
| **Error a propósito** | Una celda que revienta adrede, con la explicación del mensaje. El error es contenido, no accidente |
| **Pregunta** | Una duda que sale siempre en esta clase, con su respuesta |
| **Pregunta de interpretación** | Le toca a usted. Se responde en español, no con código, y la respuesta esperada está plegada debajo |

## El mapa del cuaderno

| Parte | Secciones | De qué va | Preguntas |
|-------|-----------|-----------|-----------|
| **A · Los datos** | 1 a 3 | Cargar, convertir la fecha, entender por qué un outlier arruina un dashboard | 1 a 5 |
| **B · Plotly** | 4 a 7 | Los tres gráficos: línea, barras y caja. Y el KPI con su ancla | 6 a 10 |
| **C · Streamlit** | 8 | Se cierra el notebook y se abre una terminal | 11 y 12 |

La parte C es la única de todo el semestre que no ocurre dentro de Jupyter. Aparecen la terminal, un
proceso que queda corriendo, un puerto y un navegador. Vale la pena avisarlo antes de llegar ahí.

---

# Parte A · Los datos

## Sección 1 · El dataset, antes de tocarlo

**Concepto.** Antes de graficar nada hay que saber qué mide cada columna y **contra qué se compara**.
Un número sin ancla no dice nada: 31,4 no es alto ni bajo hasta que alguien diga cuál es el límite.

| Columna | Qué es |
|---------|--------|
| `Municipio` | Dosquebradas, Pereira, Santa Rosa de Cabal, La Virginia |
| `Estacion` | La estación física que tomó la medida |
| `Fecha` | Texto con formato `MM/DD/YYYY hh:mm:ss AM`. Todavía no es una fecha |
| `Diametro aerodinamico` | `PM10` o `PM2.5`, el tamaño de la partícula en micrómetros |
| `Medicion` | Concentración, en microgramos por metro cúbico (`ug/m3`) |

**Los términos, la primera vez que aparecen:**

- **PM10** y **PM2.5** (*particulate matter*, material particulado): partículas suspendidas de menos
  de 10 y de 2,5 micrómetros. Las PM2.5 son tan pequeñas que pasan del pulmón al torrente sanguíneo.
- **Guía de la OMS**: 15 ug/m3 de PM2.5 como promedio diario, 45 ug/m3 de PM10. Ese es el ancla de
  toda la clase.

In [ ]:
import pandas as pd
import plotly.express as px

# La ruta: este notebook vive en clase10/demo/. El '..' sube a clase10/ y de ahí entra a data/.
df = pd.read_csv("../datos/calidad_aire_risaralda.csv")

print("Filas y columnas:", df.shape)
df.head()

In [ ]:
# ¿Y las últimas filas? tail muestra el final del archivo. Aquí importa más que de costumbre:
# la columna Fecha es texto todavía, así que el orden del archivo NO es el orden del tiempo.
df.tail()

In [ ]:
# ¿De qué tipo es cada columna y cuántos no-nulos tiene? info responde las dos de una vez.
df.info()

In [ ]:
# ¿Los números están guardados como números? dtypes lo dice sin rodeos.
# Fecha va a salir como object: es texto, y por eso la sección 2 existe.
print(df.dtypes)

In [ ]:
# ¿Dónde hay nulos y cuántos? En un dashboard esto no es cosmético: una fila nula
# atraviesa el filtro, llega al gráfico y deja un hueco que el usuario interpreta como cero.
print(df.isna().sum())

In [ ]:
# ¿Cuántas opciones tendría cada filtro del dashboard? nunique lo responde antes de
# escribir una sola línea de Streamlit: un selector de 4 opciones y otro de 400 no se
# construyen igual.
for columna in ["Municipio", "Estacion", "Diametro aerodinamico"]:
    print(f"{columna}: {df[columna].nunique()} valores distintos")

In [ ]:
# value_counts cuenta cuántas filas hay por valor. describe resume una columna numérica.
# Las dos se vieron en la clase 2 y en la clase 4.
print(df["Municipio"].value_counts())
print()
print(df["Diametro aerodinamico"].value_counts())
print()
print(df["Medicion"].describe())

### Lo que ya se ve mal, sin haber graficado nada

1. **La cobertura es despareja.** Dosquebradas tiene 1.980 registros y La Virginia 743. Y La Virginia
   no tiene PM2.5. Cuando alguien filtre por PM2.5, ese municipio va a desaparecer del gráfico y va a
   pensar que la app está rota.
2. **El máximo es 45.839,85** y la mediana está por debajo de 30. Un valor así en un sensor de aire
   no es contaminación extrema: es un error de unidades.
3. **El mínimo es 0.** Un sensor no mide cero material particulado. Cero es sensor apagado, no aire
   limpio.
4. **`Municipio` y `Estacion` son casi la misma columna:** cada municipio tiene exactamente una
   estación. **Dos filtros que hacen lo mismo cuentan como uno solo**, y ese es exactamente el error
   que se comete después con el dataset propio.

> **Para entender qué está pasando.** Esto no es limpieza por gusto: es la clase 3 aplicada a un
> dashboard. La diferencia con la clase 3 es la consecuencia. Allá un valor sucio ensuciaba una
> tabla que usted miraba; aquí ensucia una app que **otra persona** va a usar sin saber que el dato
> está sucio. En un informe estático usted decide qué se ve; en un dashboard decide el usuario.

**Pregunta de interpretación 1.** De las cuatro cosas de arriba, dos son datos sucios y dos son
problemas de **diseño del tablero**. ¿Cuáles son cuáles, y qué le va a pasar al usuario de la app si
las dos de diseño no se resuelven antes de publicarla?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Datos sucios: el máximo de 45.839,85 y el mínimo de 0. Los dos son valores que no pueden ser ciertos y
se resuelven filtrando, con la decisión escrita.

Problemas de diseño: la cobertura despareja y el par `Municipio` / `Estacion`. Ninguno de los dos es
un error del archivo —los datos son correctos— pero los dos rompen la app.

Qué le pasa al usuario. Con la cobertura despareja, filtra por PM2.5, La Virginia desaparece del
gráfico y concluye una de dos cosas: que en La Virginia no hay PM2.5 (falso: no hay *sensor*), o que
la app está rota. Las dos conclusiones son culpa suya, no de él: el tablero le dejó inferir algo que
los datos no dicen. Se arregla con una frase visible que diga qué estaciones miden qué.

Con `Municipio` y `Estacion`, el usuario ve dos filtros, gasta tiempo entendiendo en qué se
diferencian, y descubre que en nada. **Dos filtros que hacen lo mismo cuentan como uno solo**, y es
exactamente el error que se comete con el dataset propio: se llenan los tres filtros del requisito con
tres versiones de la misma columna.

</details>

## Sección 2 · Convertir la fecha

**Concepto.** `pd.to_datetime` convierte texto a fecha. El parámetro `format` describe cómo está
escrita la cadena, código por código:

| Código | Qué representa |
|--------|----------------|
| `%m` | mes en dos dígitos |
| `%d` | día en dos dígitos |
| `%Y` | año en cuatro dígitos |
| `%I` | hora en formato de 12 horas |
| `%M` | minutos |
| `%S` | segundos |
| `%p` | AM o PM |

`errors="coerce"` convierte en `NaT` (*not a time*, el nulo de las fechas) lo que no se pueda
interpretar, en vez de reventar. Se prefiere en un dashboard: es mejor perder tres filas que tener
una app que no levanta.

**Por qué importa que sea fecha de verdad y no texto:** una columna de fecha trae el accesorio `.dt`
(`.dt.year`, `.dt.month`), se puede ordenar cronológicamente y se puede agrupar por mes. Como texto,
`"09/2020"` va después de `"01/2021"` en orden alfabético, y el gráfico de línea sale al revés.

In [ ]:
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

print("Desde:", df["Fecha"].min(), " Hasta:", df["Fecha"].max())
print("Fechas que no se pudieron convertir:", df["Fecha"].isna().sum())
print("Tipo de la columna ahora:", df["Fecha"].dtype)

### Error a propósito · el `format` equivocado

La celda de abajo usa el formato **europeo** (`día/mes/año`, hora de 24 horas) sobre un archivo
escrito en formato **estadounidense** (`mes/día/año`, hora de 12 horas con AM/PM). Va a fallar. Lea
el mensaje: es el que le va a salir a usted con el CSV de su equipo.

In [ ]:
# Se relee la columna cruda, tal como venía en el archivo, para provocar el error.
fecha_cruda = pd.read_csv("../datos/calidad_aire_risaralda.csv", usecols=["Fecha"])["Fecha"]

try:
    pd.to_datetime(fecha_cruda, format="%d/%m/%Y %H:%M:%S")
except ValueError as error:
    print("ValueError:", error)
    print()
    print("Qué dice: sobró texto sin interpretar (' AM'), porque %H es hora de 24 horas y no")
    print("espera AM/PM. Y aunque quitara eso, %d/%m leería 06/09 como 6 de septiembre en vez")
    print("de 9 de junio: los datos entrarían, mal, sin lanzar ningún error.")

> **Para entender qué está pasando.** El peor caso no es este error: es cuando el formato
> equivocado **sí** funciona. `03/04/2020` es válido leído como 3 de abril y como 4 de marzo. Pandas
> no tiene forma de saber cuál es, y sin `format` adivina fila por fila. El error de arriba es un
> regalo: falla ruidosamente. La versión silenciosa de ese mismo error le desordena la serie de
> tiempo y no se entera nunca.

### El periodo cubierto, ahora que la fecha es fecha

Una columna de tipo fecha trae el accesorio `.dt`: `serie.dt.year` devuelve el año de cada fila.
Sirve para lo que hay que hacer siempre antes de graficar una serie de tiempo: saber **desde cuándo y
hasta cuándo** hay dato, y si las dos series cubren el mismo periodo.

In [ ]:
anio_primero = int(df["Fecha"].dt.year.min())
anio_ultimo = int(df["Fecha"].dt.year.max())

print(f"El archivo va de {anio_primero} a {anio_ultimo}.")

# Y ahora lo importante: cada partícula por separado.
for particula, grupo in df.groupby("Diametro aerodinamico"):
    print(f"{particula}: desde {grupo['Fecha'].min().date()} hasta {grupo['Fecha'].max().date()}",
          f"({len(grupo):,} mediciones)")

**Pregunta de interpretación 2.** El archivo empieza en 2007, pero el PM2.5 no. Escriba el
título que **no** se puede poner en el gráfico de la serie de tiempo, y explique por qué es falso
aunque salga de datos verdaderos.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

El título prohibido es cualquiera con "en todo el periodo": *"El PM10 duplica al PM2.5 en todo el
periodo"*. Los números que lo sostienen son correctos, y aun así la frase afirma algo sobre once años
en los que **no existía el sensor de PM2.5**. No es que el PM2.5 fuera bajo entre 2007 y 2012: es que
no se midió.

Es el error más caro de esta clase porque no lo detecta ningún programa. El código corre, el gráfico
sale bonito y el título miente. El único filtro es mirar el periodo de cada serie antes de escribir la
frase, que es exactamente lo que acaba de hacer la celda de arriba.

La versión honesta dice las dos cosas: *"El PM10 duplica al PM2.5, pero solo hay PM2.5 desde 2012"*.
Un título que trae su propia salvedad no es un título débil: es el que sobrevive a la primera pregunta
en la sustentación.

</details>

## Sección 3 · Por qué un outlier arruina un dashboard

**Concepto.** Un **outlier** es un valor que se sale del rango en el que vive el resto de los datos.
En la clase 4 fue la jirafa en el parque de perros: el promedio de altura del parque no le sirve a
nadie, porque hay una jirafa. Hoy la consecuencia es otra y es peor: **un solo valor extremo aplasta
la escala del eje y vuelve ilegible todo lo demás**.

Antes de limpiar nada, hay que ver el daño. El gráfico de abajo es el mismo que se va a hacer bien
en un momento, pero con los datos crudos.

In [ ]:
serie_sucia = (
    df.groupby([pd.Grouper(key="Fecha", freq="ME"), "Diametro aerodinamico"], as_index=False)
    ["Medicion"].mean()
)

fig_sucia = px.line(
    serie_sucia,
    x="Fecha",
    y="Medicion",
    color="Diametro aerodinamico",
    title="Sin limpiar: cuatro filas de 5.047 vuelven ilegible todo el resto",
)
fig_sucia.show()

Un pico y una raya plana. Diecisiete años de información aplastados por **cuatro** filas.

Y en la app va a ser peor, porque el usuario no tiene forma de saber que la raya plana esconde algo.

**Pregunta de interpretación 3.** En la clase 4 el outlier era un problema de la
**estadística**: la jirafa dañaba el promedio del parque de perros. Aquí es un problema del
**tablero**. ¿Cuál es exactamente la diferencia, y por qué el segundo es más grave?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

En la clase 4 el outlier contaminaba un número, y había defensa: se usa la mediana, se reporta el
rango, se dice cuántos outliers hay. El daño estaba acotado a un estadístico que usted controlaba.

Aquí el outlier destruye **la escala del eje**, que es el canal por el que el lector recibe toda la
información del gráfico. No daña un número: daña los diecisiete años que están dibujados al lado. Con
el eje llegando a 410, la diferencia entre un mes de 20 y uno de 45 ug/m3 ocupa dos píxeles y es
literalmente invisible.

Y es más grave por quién lo mira. Un número contaminado lo revisa usted, que sabe que existe la
jirafa. Un eje aplastado lo mira **otra persona**, en una app, sin saber que hay cuatro filas raras
detrás. El lector no ve una advertencia: ve una raya plana y concluye que el aire de Risaralda no
cambió en diecisiete años. Es una conclusión falsa producida por un gráfico técnicamente correcto.

</details>

In [ ]:
# Los sospechosos: las mediciones más altas del archivo.
df.nlargest(6, "Medicion")[["Municipio", "Fecha", "Diametro aerodinamico", "Medicion"]]

Cuatro mediciones de PM2.5 en Dosquebradas: 45.839, 37.507, 8.347 y 4.358. La mediana de
PM2.5 en ese mismo municipio es **12,7**. Son tres órdenes de magnitud de diferencia: casi con
seguridad nanogramos reportados como microgramos.

**La decisión y su justificación, que hay que escribir siempre:** se descartan las mediciones por
encima de 500 ug/m3 y las iguales a cero. No se "arreglan" ni se imputan: se excluyen, se cuenta
cuántas fueron y queda documentado en el código y en la app.

> **Para entender qué está pasando.** ¿Por qué excluir y no imputar? Porque imputar es inventar un
> valor, y aquí no se sabe cuál era el verdadero: 45.839 podría ser 45,8 (nanogramos mal reportados)
> o podría ser un sensor dañado. Inventar el número correcto sería fabricar el dato que sostiene la
> conclusión. Excluir y decir cuántas se excluyeron es honesto y se puede auditar. La regla del
> curso desde la clase 3: **nunca se descarta en silencio**.

In [ ]:
LIMITE_MEDICION = 500

filas_antes = len(df)

df_limpio = df.dropna(subset=["Fecha", "Medicion"])
df_limpio = df_limpio[
    (df_limpio["Medicion"] > 0) & (df_limpio["Medicion"] <= LIMITE_MEDICION)
].copy()

print(f"Antes:     {filas_antes}")
print(f"Después:   {len(df_limpio)}")
print(f"Se fueron: {filas_antes - len(df_limpio)} filas "
      f"({(filas_antes - len(df_limpio)) / filas_antes * 100:.2f}%)")

### El número que va escrito en la app

Cuántas filas se descartaron no es un dato interno: **va visible en el tablero**. Un dashboard que
descarta datos sin decir cuáles ni cuántos no es confiable, y en la sustentación del Momento 2 es la
primera pregunta que aparece.

In [ ]:
filas_descartadas = len(df) - len(df_limpio)

print(f"Filas descartadas: {filas_descartadas} de {len(df):,} "
      f"({filas_descartadas / len(df) * 100:.2f}%)")

# Y el desglose, que es lo que se escribe en la app: cuáles y por qué.
print("  - por encima de 500 ug/m3:", (df["Medicion"] > LIMITE_MEDICION).sum())
print("  - iguales a cero:", (df["Medicion"] == 0).sum())

**Pregunta de interpretación 4.** Trece filas de 5.047 son el 0,26% del archivo. Un compañero
dice que un cuarto de punto porcentual no merece una frase en el tablero, que es ruido. ¿Qué le
responde?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Que el porcentaje del archivo no mide el efecto en las conclusiones, y esas trece filas son la prueba:
con ellas dentro, el gráfico de la sección anterior era una raya plana ilegible, y —como se ve dos
secciones más abajo— cambian **cuál municipio encabeza el ranking**. Trece filas de 5.047 deciden el
titular del tablero. Un dato que cambia el titular no es ruido por pequeño que sea.

La segunda parte de la respuesta es de confianza, no de estadística. Quien lee el tablero no puede
auditar lo que no ve. Si el número aparece —"se descartaron 13 mediciones imposibles de 5.047"— el
lector puede evaluar la decisión. Si no aparece, tiene que confiar a ciegas, y en la sustentación esa
es la primera pregunta del jurado.

Y por eso se **excluye** en vez de imputar. Imputar es inventar el valor verdadero, y aquí nadie sabe
cuál era: 45.839,85 pudo ser 45,8 mal escrito o un sensor dañado. Inventar el número sería fabricar el
dato que sostiene la conclusión. Excluir, contar y decirlo es honesto y se puede auditar. La regla de
la clase 3, intacta: **nunca se descarta en silencio**.

</details>

---

# Parte B · Plotly

## Sección 4 · `px.line` — cómo evoluciona en el tiempo

**Concepto.** `px` es `plotly.express`, la interfaz corta de Plotly. Le entra un DataFrame y le sale
una **figura interactiva**: se le hace zoom, muestra los valores al pasar el mouse (*hover*) y las
series se apagan haciendo clic en la leyenda. Eso es lo que matplotlib no hace, y es toda la razón
por la que aparece una librería nueva a estas alturas del curso.

| Parámetro | Qué hace |
|-----------|----------|
| `x`, `y` | Qué columna va en cada eje |
| `color` | Parte los datos en una serie por cada valor de esa columna |
| `title` | El título. Y el título es el mensaje, igual que en la clase 9 |

**`px` no agrega por usted.** Si le pasa el DataFrame crudo, dibuja una línea por cada fila del
archivo. El `groupby` de la clase 4 va **antes**, siempre. Esta frase se repite tres veces en esta
clase a propósito.

**`pd.Grouper(key="Fecha", freq="ME")`** agrupa por mes calendario (*month end*). Es la forma de
decir "una fila por mes" cuando la columna es de tipo fecha.

**Lea el título de la figura que sale abajo y después mire la figura.** Dice dos cosas y las dos
importan: que el PM10 promedia el doble que el PM2.5 (29,5 contra 14,6 en los meses en que se miden
los dos), y que la serie de PM2.5 **no existe antes de septiembre de 2012**. Un título que dijera
"en todo el periodo" estaría afirmando algo sobre once años en los que ese sensor no estaba puesto.

In [ ]:
serie_mensual = (
    df_limpio.groupby(
        [pd.Grouper(key="Fecha", freq="ME"), "Diametro aerodinamico"], as_index=False
    )["Medicion"].mean()
)

fig_linea = px.line(
    serie_mensual,
    x="Fecha",
    y="Medicion",
    color="Diametro aerodinamico",
    title="El PM10 duplica al PM2.5, pero solo hay PM2.5 desde 2012",
)

# update_layout ajusta el envoltorio: ejes, leyenda, márgenes.
# El eje x va sin etiqueta a propósito: "Fecha" sobre unas fechas no le aporta nada a nadie.
fig_linea.update_layout(
    xaxis_title="",
    yaxis_title="Promedio mensual (ug/m3)",
    legend_title="",
)

fig_linea.show()

**Pregunta de interpretación 5.** Este gráfico es interactivo: se le hace zoom, muestra el
valor al pasar el mouse y las series se apagan desde la leyenda. Apague el PM10 haciendo clic en la
leyenda y mire lo que queda. ¿Qué le permite hacer al lector esta figura que la misma figura en un PDF
no le permitiría? ¿Y en qué caso esa libertad es un riesgo para usted?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Le permite **hacerse sus propias preguntas**. En un PDF usted decide qué se ve y el lector solo puede
aceptar o desconfiar. Aquí él puede aislar el PM2.5, hacer zoom en 2020 y ver el hueco de la pandemia,
o comparar dos años concretos sin pedirle a usted otro gráfico. Eso es lo único que justifica montar un
tablero en vez de exportar un PDF: si nadie va a explorarlo, el PDF es mejor, más barato y se imprime.

El riesgo es la otra cara: **el mensaje tiene que sobrevivir a todas las vistas que el lector se pueda
fabricar**. Usted escribió el título mirando la serie completa, pero él va a ver combinaciones que
usted nunca miró. Si aísla el PM2.5 y hace zoom entre 2018 y 2021, ve una caída que contradice el
título. Ahí el tablero se le vuelve en contra.

De ahí la regla de trabajo del bloque: antes de publicar, muévalo usted mismo hasta encontrar la
combinación que rompe el mensaje. O el título se corrige, o el filtro que produce esa vista no debería
existir.

</details>

### Error a propósito · el nombre de la columna

El error más frecuente de la clase, y el que más tiempo cuesta en el reto: escribir el nombre de la
columna como uno cree que se llama, y no como se llama. Lea el mensaje completo de abajo: Plotly
hace algo que pandas no hace, y es **listarle los nombres válidos**.

In [ ]:
try:
    px.line(serie_mensual, x="fecha", y="Medicion")
except ValueError as error:
    print("ValueError:", error)
    print()
    print("La columna se llama 'Fecha', con F mayúscula. Plotly le dice cuáles existen.")
    print("Por eso en el reto lo primero que se hace es renombrar TODAS las columnas a")
    print("minúsculas, sin tildes ni espacios, una sola vez, en la carga.")

## Sección 5 · `px.bar` — quién tiene el aire más cargado

**Concepto.** Mismo patrón: agregar primero, graficar después. Y `sort_values` no es cosmético: un
gráfico de barras ordenado responde "¿quién es el peor?" de un vistazo; uno desordenado obliga a leer
todas las etiquetas. Es la regla 4 de la clase 8.

**`as_index=False`** deja la columna de agrupación como columna normal en vez de convertirla en
índice. Plotly necesita que sea columna para poder ponerla en un eje.

### La tabla por municipio, limpia y sucia

Las dos versiones, una al lado de la otra. La diferencia entre ellas es el contenido de la sección.

In [ ]:
promedio_municipio = (
    df_limpio.groupby("Municipio", as_index=False)["Medicion"]
    .mean()
    .sort_values("Medicion", ascending=False)
)

print("Con los datos limpios:")
print(promedio_municipio.to_string(index=False))

print()
print("Con los datos crudos, sin descartar las trece filas:")
print(
    df.groupby("Municipio", as_index=False)["Medicion"]
    .mean()
    .sort_values("Medicion", ascending=False)
    .to_string(index=False)
)

### El gráfico de barras, y el título que lo acompaña

**El título tiene que decir una conclusión, no una etiqueta:**

- Etiqueta: "Promedio de medición por municipio". Describe los ejes, que ya se ven.
- Mensaje: "Santa Rosa de Cabal encabeza por 0,4 ug/m3: un empate técnico". Dice algo.

La figura de abajo viene con título, con el eje y etiquetado con su unidad y con el eje x en blanco a
propósito. **Ese es el modelo a copiar en el reto**, donde sí se exige y sí se califica. Que el título
sea *verdadero* no lo puede revisar ningún programa: eso lo juzga usted, y es lo que más pesa.

> **Para entender qué está pasando.** Mire `promedio_municipio` antes de escribir el título, y
> después vuelva a mirarlo agrupando sobre `df` en vez de sobre `df_limpio`. **Con los datos crudos
> Dosquebradas promedia 78,6 ug/m3 y gana por goleada; con los datos limpios promedia 30,2 y queda
> segundo, 0,4 por debajo de Santa Rosa de Cabal.** Cuatro filas de 5.047 cambian quién encabeza el
> gráfico, y por lo tanto cambian el título.
>
> Esa es la trampa del dashboard: **el título afirma un hallazgo, pero el hallazgo depende de
> decisiones de limpieza que el lector no ve.** Por eso la limpieza va escrita en la app y por eso
> aquí el título dice "empate técnico" en vez de coronar a un ganador que se decide por 1%. Un
> título que se cae si alguien mueve el límite de 500 no es un hallazgo: es una casualidad.

In [ ]:
fig_barras = px.bar(
    promedio_municipio,
    x="Municipio",
    y="Medicion",
    title="Santa Rosa de Cabal encabeza por 0,4 ug/m3: un empate técnico",
)
fig_barras.update_layout(xaxis_title="", yaxis_title="Promedio (ug/m3)")
fig_barras.show()

**Pregunta de interpretación 6.** Con los datos crudos Dosquebradas promedia 78,6 ug/m3 y
gana por goleada; con los datos limpios promedia 30,2 y queda segundo, 0,4 por debajo de Santa Rosa de
Cabal. ¿Por qué el título dice "empate técnico" en vez de coronar a Santa Rosa de Cabal?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque 0,4 ug/m3 sobre 30 es una diferencia del 1%, y hay al menos tres razones para no apostarle:
Santa Rosa de Cabal tiene 817 mediciones y Dosquebradas 1.980, la mezcla de PM10 y PM2.5 no es la
misma en los dos, y el resultado depende de dónde se puso el corte de 500. Mueva el límite y el orden
puede cambiar. **Un hallazgo que se cae si alguien toca un parámetro de limpieza no es un hallazgo: es
una casualidad.**

Lo que sí es un hallazgo, y es más grande que el ranking: **cuatro filas de 5.047 deciden quién
encabeza el gráfico.** Con los datos crudos el titular habría sido "Dosquebradas contamina 2,5 veces
más que sus vecinos", y habría sido falso.

Aquí es donde se cierra la clase 9. Allá se enseñó que el título afirma un hallazgo con su cifra. Hoy
se ve el costado incómodo de esa regla: **el hallazgo depende de decisiones de limpieza que el lector
del tablero nunca ve.** Por eso la limpieza va escrita en la app, y por eso un título honesto puede
tener que decir "empate" cuando lo que hay es un empate.

</details>

## Sección 6 · `px.box` — lo que el promedio esconde

**Concepto.** Una **caja** (*box plot*) resume una distribución en cinco números: mínimo, primer
cuartil, mediana, tercer cuartil y máximo. Los puntos sueltos por fuera de los bigotes son los
outliers que quedaron después de limpiar.

**Por qué caja y no histograma:** el histograma muestra la forma completa de **una** variable; la
caja permite comparar **muchos grupos** lado a lado. Para cuatro estaciones, caja. Para entender una
sola variable a fondo, histograma.

Vuelve la jirafa de la clase 4: dos municipios pueden tener el mismo promedio y una realidad
completamente distinta, uno estable y el otro con picos. La barra no distingue esos dos casos. La
caja sí.

**Y aquí pasa exactamente eso.** Balalaika (Dosquebradas) y Centro Urbano (Santa Rosa de Cabal)
promedian casi lo mismo, 30,2 contra 30,6, y en el gráfico de barras son dos palitos gemelos. En la
caja no se parecen: Balalaika llega a 116 ug/m3 y Centro Urbano no pasa de 75. **Mismo promedio, dos
problemas de salud pública distintos.** El promedio no era el hallazgo; la dispersión sí.

### La caja por estación

`color=` es lo que produce dos cajas por estación, una por partícula. Sin él saldría una sola caja que
mezcla PM10 con PM2.5, que son dos cosas distintas medidas en la misma unidad: el peor tipo de gráfico,
el que se puede leer y está mal.

In [ ]:
fig_caja = px.box(
    df_limpio,
    x="Estacion",
    y="Medicion",
    color="Diametro aerodinamico",
    title="Mismo promedio, distinta dispersión: Balalaika es la más variable",
)
fig_caja.update_layout(xaxis_title="", yaxis_title="Medición (ug/m3)", legend_title="")
fig_caja.show()

**Pregunta de interpretación 7.** ¿Qué muestra la caja que las barras del gráfico anterior no
dejaban ver? Y la que importa de verdad: si tuviera que recomendarle a la autoridad ambiental **dónde
intervenir primero**, ¿le sirve más la barra o la caja?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La barra muestra un solo número por estación, el promedio, y en el promedio Balalaika (Dosquebradas,
30,2) y Centro Urbano (Santa Rosa de Cabal, 30,6) son dos palitos gemelos. La caja muestra la
**distribución completa**, y ahí dejan de parecerse: Balalaika llega a 116 ug/m3 y Centro Urbano no
pasa de 75. Mismo promedio, dos realidades distintas.

Para decidir dónde intervenir, la caja, y por una razón que no es estética. Un promedio de 30 con
picos de 116 y un promedio de 30 estable **son dos problemas de salud pública diferentes**. El primero
son episodios: días concretos en que el aire es peligroso, y la respuesta es una alerta temprana o una
restricción los días críticos. El segundo es exposición crónica de fondo, y la respuesta es
estructural: fuentes fijas, tráfico, planeación.

Recomendar lo mismo para los dos por tener el mismo promedio sería el error, y **la barra no le da
manera de notarlo**. Es la jirafa de la clase 4 otra vez: el promedio no es el hallazgo, la dispersión
sí. En su tablero, cuando una barra y una caja compitan por el mismo espacio, la pregunta no es cuál se
ve mejor, es cuál cambia la decisión.

</details>

### Los tres gráficos, y las tres preguntas que responden

Un dashboard no es una colección de gráficos: es una colección de respuestas.

| Gráfico | Pregunta que responde | Decisión que habilita |
|---------|----------------------|----------------------|
| Línea | ¿Está mejorando o empeorando con los años? | Si la política actual está funcionando |
| Barras | ¿Dónde está peor el aire? | Dónde poner la próxima estación o la próxima restricción |
| Caja | ¿Qué tan estable es cada estación? | Si el problema es crónico o son picos puntuales |

**Tres gráficos que responden la misma pregunta cuentan como uno solo.** Es el criterio del reto y el
del Momento 2.

> **Pregunta.** ¿Puedo seguir usando matplotlib dentro de Streamlit?
> **Respuesta.** Sí, con `st.pyplot(fig)`. Pero pierde el hover y el zoom, que es justamente lo que
> justifica hacer un dashboard en vez de un PDF. Para un informe impreso, matplotlib y seaborn
> siguen siendo mejores: la clase 8 no queda derogada.

**Pregunta de interpretación 8.** Mire la tabla de arriba y suponga que le obligan a
**borrar uno** de los tres gráficos. ¿Cuál borra, y qué decisión deja de poder tomar quien use el
tablero?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No hay una única respuesta correcta, pero sí una forma correcta de responder: **el criterio es la
decisión que se pierde, no el gráfico que se ve peor.**

Un argumento defendible: se borra la barra. Es la que menos aporta, porque son cuatro municipios con
una diferencia del 1% entre los dos primeros —el "empate técnico" del título— y porque la caja ya
ordena las estaciones por nivel *y además* muestra la dispersión. Lo que se pierde es la lectura
inmediata de "quién está peor" de un vistazo, que para un público no técnico vale bastante.

Otro argumento igual de defendible: se borra la caja, porque exige explicar qué son los cuartiles ante
una audiencia que no los conoce, y en un tablero público eso es fricción real.

Lo que **no** se puede borrar es la línea: es el único gráfico que responde si la situación mejora o
empeora, y esa es la pregunta que hace quien financia la política. Ninguno de los otros dos la
responde ni de lejos.

Lleve el ejercicio a su propio tablero: si al borrar un gráfico no desaparece ninguna decisión, ese
gráfico estaba de adorno. Es exactamente lo que se penaliza en el Momento 2.

</details>

## Sección 7 · Un KPI necesita su ancla

**Concepto.** Un **KPI** (*key performance indicator*, indicador clave) es un número que cambia una
decisión. "Promedio: 14,6" no cambia ninguna: nadie sabe si 14,6 es bueno o malo. "Una de cada tres
mediciones de PM2.5 supera la guía de la OMS" sí, porque trae el ancla adentro.

Los tres KPIs de un dashboard salen de esa pregunta: **¿qué número haría que alguien actuara
distinto?** No: ¿qué número es fácil de calcular?

### El KPI del ancla, calculado

`(serie > umbral).mean()` devuelve la **proporción**, porque el promedio de una columna de verdaderos
y falsos es la fracción de verdaderos. Multiplicada por 100 es el porcentaje. Es el truco de pandas
que más se usa en un tablero.

In [ ]:
GUIA_OMS_PM25 = 15

pm25 = df_limpio[df_limpio["Diametro aerodinamico"] == "PM2.5"]
sobre_guia = (pm25["Medicion"] > GUIA_OMS_PM25).mean() * 100

print(f"Mediciones de PM2.5: {len(pm25):,}")
print(f"Promedio: {pm25['Medicion'].mean():.1f} ug/m3")
print(f"Por encima de la guía de la OMS ({GUIA_OMS_PM25} ug/m3): {sobre_guia:.1f}%")

**Pregunta de interpretación 9.** Los dos números salen de las mismas 1.020 mediciones:
"promedio 14,6 ug/m3" y "36,4% de las mediciones superan la guía de la OMS". El primero está por
**debajo** del límite y el segundo dice que se incumple más de un día de cada tres. ¿Se contradicen?
¿Cuál pondría en el tablero?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No se contradicen: dicen cosas distintas sobre la misma distribución. El promedio la resume en un
punto y queda por debajo de 15 porque hay muchos días bajos que compensan los altos. El porcentaje
cuenta **cuántas veces se cruzó la línea**, y cruzarla 36 veces de cada 100 no lo compensa ningún día
limpio: la exposición a un pico no se devuelve.

Es la lección de la clase 4 con consecuencia de política pública. El promedio no es la realidad, es un
resumen de la realidad, y aquí resume mal porque **la guía de la OMS es un límite diario, no un
promedio anual**. Comparar un promedio contra un umbral diario es comparar cosas distintas.

En el tablero va el porcentaje, y por una razón operativa: es el único de los dos que trae el ancla
adentro. "14,6" obliga al lector a saberse la guía de memoria para saber si eso es bueno o malo;
"36,4% por encima de la guía" ya trae la comparación hecha y **cambia lo que alguien hace mañana**.
Ese es el criterio para elegir los tres KPIs del reto: no cuál es fácil de calcular, sino cuál haría
que alguien actuara distinto.

</details>

---

# Parte C · Streamlit

## Sección 8 · De aquí a la app

Se cierra el notebook y se abre `streamlit_demo.py`.

**Ese archivo no es un notebook.** No tiene celdas, no se ejecuta con el botón de play del editor, y
no se corre con `python streamlit_demo.py`. Solo funciona así, desde la terminal:

```
cd clase10/demo
streamlit run streamlit_demo.py
```

Para detenerla: `Ctrl+C` **en la terminal**. Cerrar la pestaña del navegador no la apaga. Si el
puerto está ocupado: `streamlit run streamlit_demo.py --server.port 8502`.

> **Concepto — Plotly y Streamlit no compiten.** Plotly pinta los cuadros; Streamlit es la galería:
> las paredes, la luz, los letreros y la puerta por la que entra el visitante. Un cuadro sin galería
> se queda en el taller. Una galería sin cuadros es un salón vacío con buena luz.

## Los seis pasos del patrón

Este orden sirve para cualquier dashboard, y es el que se sigue en el reto.

| Paso | Qué se agrega | Qué hay que entender |
|------|---------------|----------------------|
| 0 | `st.set_page_config(...)` y `st.title(...)` | Con dos líneas ya hay app corriendo. Se levanta antes de tener nada |
| 1 | `cargar_datos()` con `@st.cache_data` | El concepto más importante del bloque. Está explicado abajo |
| 2 | Los tres filtros en `st.sidebar` | `st.multiselect` para lo categórico, `st.slider` para el rango |
| 3 | El filtrado con `isin` y `between` | `isin` es el de la clase 2. Lo mismo, ahora alimentado por un widget |
| 4 | Los tres KPIs con `st.columns(3)` y `st.metric` | `st.columns` devuelve contenedores; `with col1:` mete cosas adentro |
| 5 | Los gráficos con `st.plotly_chart(fig, width="stretch")` | Los `fig` son exactamente los del notebook. Copiar y pegar |
| 6 | Guardas: `st.warning` + `st.stop()` si un filtro queda vacío | Sin esto, el primer usuario que deseleccione todo tumba la app |

> **Pregunta.** ¿Por qué el archivo se vuelve a ejecutar completo cada vez que muevo un filtro?
> **Respuesta.** Porque el modelo de Streamlit es "el script es la interfaz". No hay callbacks ni
> estado por defecto: cambia una entrada, se recalcula todo de arriba a abajo. Es más simple de
> escribir y más caro de ejecutar, y de ahí sale `@st.cache_data`.

> **Pregunta.** ¿Y `@st.cache_data` qué guarda exactamente?
> **Respuesta.** El valor de retorno de la función, indexado por sus argumentos. Si se la llama otra
> vez con los mismos argumentos, no ejecuta el cuerpo. Por eso se pone en la carga del CSV, que es lo
> caro y lo que no cambia.

> **Para entender qué está pasando.** El *decorador* `@st.cache_data` es una función que envuelve a
> otra función. Escribir `@st.cache_data` encima de `def cargar_datos()` es equivalente a escribir
> `cargar_datos = st.cache_data(cargar_datos)`. No hace falta saber más hoy; lo que sí hace falta es
> saber **dónde** ponerlo: en lo caro y determinista (leer y limpiar el archivo), nunca en lo que
> depende de un widget.

**Pregunta de interpretación 10.** Levante la app con `streamlit run streamlit_demo.py`,
mueva un filtro y mire el archivo `.py` mientras tanto. Sabiendo que el script entero se vuelve a
ejecutar en cada movimiento, ¿qué **no** hay que poner nunca dentro de una función con
`@st.cache_data`, y qué pasaría si lo pone?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Nada que dependa de un widget, y nada que tenga que cambiar con el tiempo o con quién mire.

Si dentro de la función cacheada mete el filtrado —o sea, si le pasa la selección del `multiselect` y
guarda el resultado— Streamlit va a devolver el DataFrame de la primera llamada mientras los
argumentos no cambien. Con argumentos que sí cambian no hay error, pero se llena la memoria con una
copia del DataFrame por cada combinación de filtros que alguien pruebe. El síntoma es el peor
posible: **la app funciona, y va cada vez más lenta**.

El caso realmente feo es cachear algo con dependencias que Streamlit no ve, como la hora actual o el
contenido de un archivo que otro proceso reescribe. La caché no tiene forma de saber que el mundo
cambió, y el tablero muestra dato viejo sin avisar. Nadie ve un error; ven un número que ya no es
cierto.

La regla, corta: **la caché va en lo caro y determinista** —leer el CSV, convertir fechas, limpiar—,
que es exactamente una vez por sesión. Todo lo que depende de un control se calcula de nuevo en cada
ejecución, y está bien que así sea: es barato, porque opera sobre un DataFrame que ya está en memoria.

</details>

**Pregunta de interpretación 11.** En la app del demo, deseleccione **todos** los
municipios y mire qué pasa antes de leer las guardas. ¿Por qué un filtro vacío no es un caso raro que
se pueda dejar para después?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque no es un caso raro: es el primer clic de mucha gente. Quien abre un tablero con todo
seleccionado y quiere ver un solo municipio suele deseleccionar todo primero y después marcar el que
le interesa. En ese instante intermedio el DataFrame filtrado queda vacío, y sin guarda todo lo que
viene después opera sobre nada: el promedio da `NaN`, el máximo revienta, el gráfico sale en blanco o
la app muestra un traceback rojo en pantalla.

Y ahí está la diferencia con el notebook, que es el punto de la sección. En el cuaderno un error lo ve
usted, que sabe qué hizo y puede volver atrás. En la app el error lo ve **otra persona**, en su
pantalla, sin contexto y sin poder arreglarlo. Un tablero que revienta delante de quien tenía que
tomar la decisión no queda en "tiene un bug": queda en que no se puede usar, y esa impresión no se
recupera.

Por eso la guarda no es un detalle de robustez: es parte de la interfaz. `st.warning` dice qué pasó en
español, `st.stop()` corta la ejecución ahí mismo y el usuario ve una instrucción en vez de un
traceback. Dos líneas, y es el error que más aparece en el reto.

</details>

### Si algo falla al levantar la app

| Síntoma | Causa | Solución |
|---------|-------|----------|
| `command not found: streamlit` | No está instalado, o está en otro entorno | `pip install streamlit`, o `python -m streamlit run archivo.py` |
| `ModuleNotFoundError: No module named 'streamlit'` | Instalado en otro Python | `which python` y `which streamlit` tienen que apuntar al mismo entorno |
| `Port 8501 is already in use` | Hay otra app suya corriendo de hace diez minutos | `streamlit run archivo.py --server.port 8502` |
| Corre el `.py` con el botón de play y no pasa nada | Streamlit no se ejecuta como script normal | Solo `streamlit run archivo.py`, desde la terminal |
| Edita el archivo y la app no cambia | Falta guardar | `Cmd+S` y el botón "Rerun" arriba a la derecha |

**Pregunta de interpretación 12.** Cierre el cuaderno y piense en el dataset de su equipo. Escriba
las **tres preguntas** que va a responder su tablero, y al lado de cada una el filtro y el gráfico que
le hacen falta. Si dos preguntas comparten gráfico, tiene un problema: dígalo.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No hay respuesta esperada: depende de su dataset. Lo que sí hay es una forma de saber si la suya
sirve.

Cada una de las tres tiene que ser una pregunta que **alguien concreto** se hace y cuya respuesta le
cambia lo que hace. "¿Cómo se distribuye la variable X?" no lo es: no hay nadie del otro lado. "¿En
qué mes conviene programar el mantenimiento?" sí.

Si dos preguntas se responden con el mismo gráfico, no son dos preguntas: es una, y le falta encontrar
la otra. Ese es el modo de falla más común del Momento 2, y se ve a un kilómetro en la sustentación:
tres gráficos que dicen lo mismo con tres formas distintas.

Y si un filtro no aparece al lado de ninguna pregunta, ese filtro sobra. Cada control es una pregunta
que usted anticipó; los que no anticipan nada solo le agregan combinaciones en las que el mensaje
puede romperse.

Este ejercicio, hecho hoy en cinco minutos, es literalmente el primer paso del reto que empieza en
media hora.

</details>

## Lo que hay que llevarse

1. `px` no agrega. El `groupby` va antes, siempre.
2. Un outlier extremo no es un detalle estadístico: es lo que vuelve inservible el gráfico.
3. Un número sin su ancla no significa nada. 31,4 no dice nada; "el doble de la guía de la OMS" sí.
4. Cada gráfico responde una pregunta distinta, o sobra.
5. Streamlit reejecuta el script completo con cada interacción. De ahí sale `@st.cache_data`.
6. Sin `st.stop()`, el primer usuario que deseleccione todo tumba su app.

**Ahora en el reto:** el mismo patrón de seis pasos, primero sobre un dataset que no vio hoy y
después sobre el de su equipo. Aquí no se tecleó nada; allá se teclea todo, y **allá sí hay
verificador**. Los gráficos de este cuaderno —titulados, con los ejes etiquetados y su unidad— son el
modelo a copiar: el reto lo exige y lo comprueba.